## Libraries

In [1]:
import pandas as pd
import numpy as np
import seaborn as sb
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.metrics import classification_report,confusion_matrix

In [2]:
import warnings
warnings.filterwarnings("ignore")

## Analyze Data

In [3]:
data=pd.read_csv(r"E:\Works\NLP\Dataset\Churn_Modelling.csv")
data.head(8)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
5,6,15574012,Chu,645,Spain,Male,44,8,113755.78,2,1,0,149756.71,1
6,7,15592531,Bartlett,822,France,Male,50,7,0.00,2,1,1,10062.80,0
7,8,15656148,Obinna,376,Germany,Female,29,4,115046.74,4,1,0,119346.88,1


In [4]:
data["Exited"].value_counts()

Exited
0    7963
1    2037
Name: count, dtype: int64

In [5]:
data=data.drop(columns=["RowNumber","CustomerId","Surname"],axis=1)
data.head(8)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
5,645,Spain,Male,44,8,113755.78,2,1,0,149756.71,1
6,822,France,Male,50,7,0.00,2,1,1,10062.80,0
7,376,Germany,Female,29,4,115046.74,4,1,0,119346.88,1


In [6]:
#1
# data["Gender"].replace({"Female":0,"Male":1},inplace=True)
# data.head(8)
#2
label_endocer_gender=LabelEncoder()
data["Gender"]=label_endocer_gender.fit_transform(data["Gender"])
data.head(4)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0


In [7]:
# #1
# geo_encoded_df=pd.get_dummies(data[["Geography"]],dtype=int)
# geo_encoded_df
#2
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo=OneHotEncoder()
geo_encoder=onehot_encoder_geo.fit_transform(data[["Geography"]])
geo_encoder_df=pd.DataFrame(geo_encoder.toarray(),columns=onehot_encoder_geo.get_feature_names_out(["Geography"]))
print(onehot_encoder_geo.get_feature_names_out(["Geography"]))
print(geo_encoder_df.head(8))

['Geography_France' 'Geography_Germany' 'Geography_Spain']
   Geography_France  Geography_Germany  Geography_Spain
0               1.0                0.0              0.0
1               0.0                0.0              1.0
2               1.0                0.0              0.0
3               1.0                0.0              0.0
4               0.0                0.0              1.0
5               0.0                0.0              1.0
6               1.0                0.0              0.0
7               0.0                1.0              0.0


In [8]:
## Combine new df to older df.
data=pd.concat([data.drop("Geography",axis=1),geo_encoder_df],axis=1)
data.head(8)

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
5,645,1,44,8,113755.78,2,1,0,149756.71,1,0.0,0.0,1.0
6,822,1,50,7,0.00,2,1,1,10062.80,0,1.0,0.0,0.0
7,376,0,29,4,115046.74,4,1,0,119346.88,1,0.0,1.0,0.0


In [9]:
## Save the encoders and scaler.
import pickle

with open("label_encoder_gender.pkl","wb") as file:
    pickle.dump(label_endocer_gender,file)
    
with open("onehot_encoder_geo.pkl","wb") as file:
    pickle.dump(onehot_encoder_geo,file)

In [10]:
## Divie the dataset into indepent and dependent features.
X=data.drop("Exited",axis=1)
y=data["Exited"]

## Split the data in training and testing sets.
X_train,X_test,y_train,y_test=train_test_split(X,y,train_size=0.2,random_state=42)

## Scale these features.
Scaler=StandardScaler()
X_train_scale=Scaler.fit_transform(X_train)
X_test_scale=Scaler.transform(X_test)

In [11]:
X_train_scale

array([[-0.38381126, -1.12244688,  2.60786056, ...,  1.00400803,
        -0.57427105, -0.58350885],
       [-0.73723819, -1.12244688,  0.12136034, ..., -0.99600797,
         1.74133801, -0.58350885],
       [ 0.32304261,  0.89091075, -0.54808203, ...,  1.00400803,
        -0.57427105, -0.58350885],
       ...,
       [ 0.85318301, -1.12244688, -0.06990891, ...,  1.00400803,
        -0.57427105, -0.58350885],
       [ 0.14632915,  0.89091075,  0.40826421, ...,  1.00400803,
        -0.57427105, -0.58350885],
       [ 0.45817644,  0.89091075,  1.1733412 , ..., -0.99600797,
         1.74133801, -0.58350885]])

In [12]:
with open("Scaler.pkl","wb") as file:
    pickle.dump(Scaler,file)

## ANN Implementation

In [13]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [14]:
## Build our ANN model.
model=Sequential([
    Dense(64,activation="relu",input_shape=(X_test_scale.shape[1],)), ##H1
    Dense(32,activation="relu"), ##H2
    Dense(1,activation="sigmoid") ##output
])

In [15]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [16]:
opt=tf.keras.optimizers.Adam(learning_rate=0.01)
loss=tf.keras.losses.BinaryFocalCrossentropy()

In [17]:
## Compile our model.
model.compile(optimizer=opt,loss=loss,metrics=["accuracy"])

In [18]:
## Set up the Tensorboard
log_dir="logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tf_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [19]:
## Set up Early Stopping.
Early_stopping_callback=EarlyStopping(monitor="val_loss",patience=10,restore_best_weights=True)

In [20]:
## Train the model.
history=model.fit(X_train_scale,y_train,validation_data=(X_test_scale,y_test),epochs=100,
                callbacks=[tf_callback,Early_stopping_callback])

Epoch 1/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8025 - loss: 0.1196 - val_accuracy: 0.8250 - val_loss: 0.1064
Epoch 2/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8420 - loss: 0.1022 - val_accuracy: 0.8303 - val_loss: 0.1060
Epoch 3/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8370 - loss: 0.0959 - val_accuracy: 0.8410 - val_loss: 0.1044
Epoch 4/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8470 - loss: 0.0939 - val_accuracy: 0.8394 - val_loss: 0.1039
Epoch 5/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8550 - loss: 0.0904 - val_accuracy: 0.8522 - val_loss: 0.1002
Epoch 6/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8600 - loss: 0.0877 - val_accuracy: 0.8537 - val_loss: 0.1045
Epoch 7/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8555 - loss: 0.0864 - val_accuracy: 0.8455 - val_loss: 0.1010
Epoch 8/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8630 - loss: 0.0869 - val_accuracy: 0.8379 - v

In [21]:
model.save("model.h5")

In [22]:
%load_ext tensorboard

In [25]:
%tensorboard --logdir logs/fit/20260826-150850

Reusing TensorBoard on port 6006 (pid 6052), started 0:00:12 ago. (Use '!kill 6052' to kill it.)